[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day11-profiling-with-torch-profiler.ipynb)

# Day 11 — Profiling with torch.profiler

Today you open the ledger on your own decoder: profile a real `generate()` call, read the op table, count kernel launches per token, and compute the launch-overhead share from your own measurement. CPU works for everything; a T4 makes the CUDA columns interesting.

**Plan:** (1) build a small cached decoder, (2) time a 20-token baseline, (3) profile it and read the top-15 table, (4) export a chrome trace, (5) count launches/token and do the overhead ledger, (6) check the big GEMM's achieved FLOP/s, (7) contrast a prefill trace, (8) write your 3-bullet fix list.

In [ ]:
# One and only pip install cell
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
# Expected output: torch already installed on Colab (this is a no-op there)

## 1. A small cached decoder to profile

Self-contained: 4-layer GPT with KV caching, greedy decode. Small on purpose — profiling must be fast on CPU.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.profiler as profiler

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
torch.manual_seed(11)

class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv  = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.fc1  = nn.Linear(d, 4 * d, bias=False)
        self.fc2  = nn.Linear(4 * d, d, bias=False)
        self.heads, self.dh = heads, d // heads

    def forward(self, x, kv=None):
        B, T, D = x.shape
        qkv = self.qkv(self.ln1(x)).view(B, T, 3, self.heads, self.dh).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if kv is not None:
            k = torch.cat([kv[0], k], dim=2)
            v = torch.cat([kv[1], v], dim=2)
        att = (q @ k.transpose(-2, -1) / math.sqrt(self.dh)).softmax(-1) @ v
        att = att.permute(0, 2, 1, 3).reshape(B, T, D)
        x = x + self.proj(att)
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x))))
        return x, (k, v)

class TinyGPT(nn.Module):
    def __init__(self, vocab=1000, d=128, layers=4, heads=4):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)

    def forward(self, idx, kvs=None):
        x = self.tok(idx)
        new_kvs = []
        for i, b in enumerate(self.blocks):
            x, kv = b(x, None if kvs is None else kvs[i])
            new_kvs.append(kv)
        return self.head(self.ln(x)), new_kvs

@torch.no_grad()
def generate(model, prompt, max_new=20):
    idx, kvs = prompt, None
    for _ in range(max_new):
        logits, kvs = model(idx[:, -1:] if kvs is not None else idx, kvs)
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx

model = TinyGPT().to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"TinyGPT: {n_params/1e6:.2f}M params, 4 layers, d=128")
prompt = torch.randint(0, 1000, (1, 8), device=device)
# Expected output:
# Device: cpu (or cuda on a T4 runtime)
# TinyGPT: 1.05M params, 4 layers, d=128

## 2. Baseline timing (no profiler)

Measure wall time for 20 decode tokens first — the profiler adds its own overhead, so the honest number comes from a clean run.

In [ ]:
generate(model, prompt, max_new=2)          # warmup
t0 = time.perf_counter()
out = generate(model, prompt, max_new=20)
dt = (time.perf_counter() - t0) * 1000
print(f"20 tokens in {dt:.1f} ms -> {20/dt*1000:.1f} tok/s, {dt/20:.2f} ms/token")
# Expected output (CPU): 20 tokens in ~40-120 ms -> ~170-500 tok/s
# (tiny model; the point is the breakdown, not the absolute speed)

## 3. Profile the decode loop

Wrap the same 20-token generation. On CUDA we sort by self CUDA time; on CPU, by self CPU time. `record_shapes=True` lets you compute each GEMM's intensity by hand.

In [ ]:
acts = [profiler.ProfilerActivity.CPU]
if device == "cuda":
    acts.append(profiler.ProfilerActivity.CUDA)
sort_key = "self_cuda_time_total" if device == "cuda" else "self_cpu_time_total"

with profiler.profile(activities=acts, record_shapes=True) as prof:
    generate(model, prompt, max_new=20)

print(prof.key_averages().table(sort_by=sort_key, row_limit=15))
# Expected: aten::mm / aten::addmm at rank 1 with ~55-65% of time;
# aten::copy_, softmax / norm kernels in the top 10.

## 4. Export the chrome trace

Copy `day11_trace.json` to your laptop and open it in `chrome://tracing` (or the Perfetto UI). Look for the gaps: idle stretches on the CUDA stream while the CPU thread is busy — the visual signature of launch overhead.

In [ ]:
prof.export_chrome_trace("day11_trace.json")
print("saved day11_trace.json -> open in chrome://tracing or https://ui.perfetto.dev")
# Expected: saved day11_trace.json ...
# In Colab: download the file from the file browser, open locally.

## 5. Count launches per token: the overhead ledger

Total aten-op calls ÷ tokens = launches per token. At ~8 µs of host submission overhead each, you get the overhead share of decode time — the day's headline number.

In [ ]:
ka = prof.key_averages()
total_calls = sum(e.count for e in ka if e.key.startswith("aten::"))
TOKENS = 20
lpt = total_calls / TOKENS
overhead_ms = total_calls * 8e-3          # 8 us per launch -> ms
print(f"aten op calls: {total_calls}  ->  {lpt:.0f} launches/token")
print(f"launch overhead ~= {total_calls} x 8 us = {overhead_ms:.1f} ms")
print(f"vs baseline {dt:.1f} ms for 20 tokens -> overhead share ~= {overhead_ms/dt*100:.0f}%")
# Expected (4-layer eager toy): tens of aten calls per token
# (this block alone issues ~20 ops/layer/step), so lpt lands ~50-120/token
# and the overhead share ~30-70%. Scale to 32 layers x 70 tokens
# for the real-model version of this ledger.


## 6. Achieved FLOP/s on the biggest decode GEMM

Take the largest `aten::mm` shape from the table, compute intensity = FLOPs/bytes by hand, and compare achieved FLOP/s against your GPU's peak — the roofline, measured.

In [ ]:
# Biggest GEMM here: the MLP up-projection, (1, 128) x (128, 512), fp32
M, K, N = 1, 128, 512
flops = 2 * M * K * N
wbytes = (K * N + M * N) * 4          # fp32 weights + output
print(f"decode GEMM: {flops/1e6:.3f} MFLOP / {wbytes/1e6:.3f} MB = {flops/wbytes:.2f} FLOP/byte")
print("H100 ridge ~295 FLOP/byte -> this kernel is ~300x below the ridge: memory-bound,")
print("achieved utilization ~1-2% of peak. Compare with your trace's mm share.")
# Expected output:
# decode GEMM: 0.131 MFLOP / 0.264 MB = 0.50 FLOP/byte
# ... memory-bound, achieved utilization ~1-2% of peak.

## 7. Contrast: a prefill trace

Profile one forward pass over a 64-token prompt (prefill-like: all tokens at once). Watch `aten::mm` jump to ~85%+ of the table — compute-bound, the Day 10 roofline in trace form.

In [ ]:
prefill_prompt = torch.randint(0, 1000, (1, 64), device=device)
with profiler.profile(activities=acts) as prof2:
    with torch.no_grad():
        model(prefill_prompt)
print(prof2.key_averages().table(sort_by=sort_key, row_limit=8))
# Expected: aten::mm dominates (~85%+ of time) — prefill is compute-bound.
# Compare launches: 1 forward = ~50 launches vs ~50/token in decode.

## 8. Your 3-bullet fix list

Write these in your notes — Weeks 7–8 tick them off:

1. **Fuse the elementwise tail** (RMSNorm + residual + scaling) into one kernel — attacks the `copy_`/`add`/`norm` rows and their launches.
2. **Capture the decode step in a CUDA graph** — replays ~50 launches for ~1 µs each instead of ~8 µs: the overhead row collapses.
3. **`torch.compile` the step function** — Inductor fuses pointwise ops and rewrites the launch pattern; re-profile and compare tables.

## Wrap-up

You read a real trace today: GEMMs at ~1 FLOP/byte, launch overhead as a line item, and the visual gap between prefill and decode. Tomorrow (Day 12) you open the hood of HF `transformers` `generate()` — the same loop you just profiled, now understood from the inside.